# 01 — Database Setup

Loads all four source CSVs into a local SQLite database (`cms_data.db`).

Run this notebook once at project start. All subsequent analysis notebooks query from this database.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

In [2]:
RAW = Path("../data/raw")
DB  = Path("../cms_data.db")

FILES = {
    "inpatient_claims":         RAW / "DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv",
    "outpatient_claims":        RAW / "DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv",
    "prescription_drug_events": RAW / "DE1_0_2008_to_2010_Prescription_Drug_Events_Sample_1.csv",
}

BENE_FILES = {
    2008: RAW / "DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv",
    2009: RAW / "DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv",
    2010: RAW / "DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv",
}

## Load inpatient, outpatient, and prescription tables

In [3]:
conn = sqlite3.connect(DB)
tables_written = {}

for name, path in FILES.items():
    df = pd.read_csv(path, low_memory=False)
    df.to_sql(name, conn, if_exists="replace", index=False)
    row_count = conn.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    tables_written[name] = row_count
    print(f"{name:<30} {row_count:>10,} rows")

print()

inpatient_claims                   66,773 rows
outpatient_claims                 790,790 rows
prescription_drug_events        5,552,421 rows



## Load and stack beneficiary summary (2008 / 2009 / 2010)

In [ ]:
bene_frames = []
for year, path in BENE_FILES.items():
    df = pd.read_csv(path, low_memory=False)
    df.insert(0, "YEAR", year)
    bene_frames.append(df)

bene = pd.concat(bene_frames, ignore_index=True)
bene.to_sql("beneficiary_summary", conn, if_exists="replace", index=False)
row_count = conn.execute("SELECT COUNT(*) FROM beneficiary_summary").fetchone()[0]
tables_written["beneficiary_summary"] = row_count
print(f"{'beneficiary_summary':<30} {row_count:>10,} rows")

conn.close()

## Verify

In [ ]:
print(f"\n✓ Database created: {DB.resolve()}\n")
print("Tables written:")
for name, count in tables_written.items():
    print(f"  {name:<30} {count:>10,}")